## Task

### Ordered-vs-Shuffled window experiment (FD001 and FD002)
### Self-contained: loads data, defines add_rul, builds window
### features, runs engine-level CV, compares ordered vs shuffled.

In [2]:
# Imports

import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

In [3]:
RNG_SEED  = 42
RUL_CAP   = 125
WINDOWS   = [5, 10, 20, 30, 50]
N_REPEATS = 3
N_FOLDS   = 5
LATE_CUT  = RUL_CAP // 2

## 0. Load data with fallbacks

In [4]:
COLS = (["unit_id", "cycle", "setting_1", "setting_2", "setting_3"]
        + [f"sensor_{i}" for i in range(1, 22)])

def _load(name, ncols):
    """Try <name>.csv, then <name>.txt, then globals(). Return DataFrame or raise."""
    for ext in (".csv", ".txt"):
        path = name + ext
        if os.path.exists(path):
            if ext == ".txt":
                return pd.read_csv(path, sep=r"\s+", header=None,
                                   names=COLS if ncols == 26 else ["RUL"])
            return pd.read_csv(path)
    # fall back to an in-scope variable with the same name (e.g. from Part A)
    if name in globals():
        return globals()[name]
    raise FileNotFoundError(
        f"Could not find {name}.csv / {name}.txt and '{name}' is not defined."
    )

df_train_FD001 = _load("train_FD001", 26)
df_train_FD002 = _load("train_FD002", 26)
df_test_FD001  = _load("test_FD001",  26)
df_test_FD002  = _load("test_FD002",  26)
df_rul_FD001   = _load("RUL_FD001",    1)
df_rul_FD002   = _load("RUL_FD002",    1)

## 1. add_rul (self-contained version of the notebook helper)

In [5]:
def add_rul(df):
    df = df.copy()
    mx = df.groupby("unit_id")["cycle"].transform("max")
    df["RUL"] = mx - df["cycle"]
    return df

## 2. Window feature extraction

In [6]:
def _window_features(values, cycles):
    """Mean/std are order-invariant; slope/diff_mean/diff_std are order-dependent."""
    n = len(values)
    if n == 0:
        return dict(mean=np.nan, std=np.nan, slope=np.nan,
                    diff_mean=np.nan, diff_std=np.nan)
    mean = values.mean()
    std  = values.std(ddof=0) if n > 1 else 0.0
    if n >= 2:
        x = cycles.astype(float); y = values.astype(float)
        xm, ym = x.mean(), y.mean()
        denom = ((x - xm) ** 2).sum()
        slope = 0.0 if denom == 0 else ((x - xm) * (y - ym)).sum() / denom
        d = np.diff(y)
        diff_mean, diff_std = d.mean(), d.std(ddof=0)
    else:
        slope = diff_mean = diff_std = 0.0
    return dict(mean=mean, std=std, slope=slope,
                diff_mean=diff_mean, diff_std=diff_std)


def build_features(df, sensor_cols, L, shuffle=False, rng=None):
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    df = df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    parts = []
    for unit, g in df.groupby("unit_id", sort=False):
        g = g.reset_index(drop=True)
        n = len(g)
        feats = {f"{s}_{k}": np.full(n, np.nan)
                 for s in sensor_cols
                 for k in ("mean", "std", "slope", "diff_mean", "diff_std")}
        for i in range(n):
            lo = 0 if L in (None, 1) else max(0, i - L + 1)
            win = g.iloc[lo:i + 1]
            cyc = win["cycle"].to_numpy()
            for s in sensor_cols:
                vals = win[s].to_numpy(dtype=float)
                if shuffle and len(vals) > 1:
                    vals = rng.permutation(vals)
                f = _window_features(vals, cyc)
                for k, v in f.items():
                    feats[f"{s}_{k}"][i] = v
        block = pd.DataFrame(feats)
        block["unit_id"] = unit
        block["RUL"]     = g["RUL"].to_numpy()
        parts.append(block)
    return pd.concat(parts, ignore_index=True)

## 3. One evaluation for (dataset, L, shuffle) with a given seed

In [7]:
def evaluate(train_df, sensor_cols, L, shuffle, seed):
    rng = np.random.default_rng(seed)
    X_df = build_features(train_df, sensor_cols, L, shuffle=shuffle, rng=rng)
    feat_cols = [c for c in X_df.columns if c not in ("unit_id", "RUL")]
    X = X_df[feat_cols].to_numpy()
    y = X_df["RUL"].to_numpy()
    groups = X_df["unit_id"].to_numpy()

    gkf = GroupKFold(n_splits=N_FOLDS)
    preds = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups):
        model = RandomForestRegressor(
            n_estimators=300, min_samples_leaf=3,
            n_jobs=-1, random_state=seed,
        )
        model.fit(X[tr], y[tr])
        preds[te] = model.predict(X[te])
    return y, np.clip(preds, 0, RUL_CAP)


def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

## 4. Main loop

In [8]:
results = []

for ds_name, df in [("FD001", df_train_FD001), ("FD002", df_train_FD002)]:
    df = add_rul(df)
    sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
    nunique = df[sensor_cols].nunique()
    sensor_cols = [c for c in sensor_cols if nunique[c] > 1]

    for L in WINDOWS:
        for rep in range(N_REPEATS):
            seed = RNG_SEED + 1000 * rep + L
            y_o, p_o = evaluate(df, sensor_cols, L, shuffle=False, seed=seed)
            y_s, p_s = evaluate(df, sensor_cols, L, shuffle=True,  seed=seed)

            early = y_o > LATE_CUT
            late  = y_o <= LATE_CUT

            results.append(dict(
                dataset=ds_name, L=L, repeat=rep,
                rmse_ordered=rmse(y_o, p_o),
                rmse_shuffled=rmse(y_s, p_s),
                delta_all=rmse(y_o, p_o) - rmse(y_s, p_s),
                rmse_ordered_early=rmse(y_o[early], p_o[early]),
                rmse_shuffled_early=rmse(y_s[early], p_s[early]),
                delta_early=rmse(y_o[early], p_o[early]) - rmse(y_s[early], p_s[early]),
                rmse_ordered_late=rmse(y_o[late], p_o[late]),
                rmse_shuffled_late=rmse(y_s[late], p_s[late]),
                delta_late=rmse(y_o[late], p_o[late]) - rmse(y_s[late], p_s[late]),
            ))

res = pd.DataFrame(results)

KeyboardInterrupt: 

## 5. Reference points: L=1 (no history) and mean-RUL predictor

In [ ]:
ref_rows = []
for ds_name, df in [("FD001", df_train_FD001), ("FD002", df_train_FD002)]:
    df = add_rul(df)
    sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
    nunique = df[sensor_cols].nunique()
    sensor_cols = [c for c in sensor_cols if nunique[c] > 1]

    for rep in range(N_REPEATS):
        seed = RNG_SEED + 1000 * rep
        y1, p1 = evaluate(df, sensor_cols, L=1, shuffle=False, seed=seed)
        pred_mean = np.full_like(y1, y1.mean(), dtype=float)
        early = y1 > LATE_CUT
        late  = y1 <= LATE_CUT

        ref_rows.append(dict(dataset=ds_name, repeat=rep, L=1,
                             rmse_ordered=rmse(y1, p1),
                             rmse_ordered_early=rmse(y1[early], p1[early]),
                             rmse_ordered_late=rmse(y1[late], p1[late])))
        ref_rows.append(dict(dataset=ds_name, repeat=rep, L="mean",
                             rmse_ordered=rmse(y1, pred_mean),
                             rmse_ordered_early=rmse(y1[early], pred_mean[early]),
                             rmse_ordered_late=rmse(y1[late], pred_mean[late])))

ref = pd.DataFrame(ref_rows)

## 6. Summary tables

In [ ]:
summary = (res.groupby(["dataset", "L"])
              .agg(delta_all_mean =("delta_all",   "mean"),
                   delta_all_std  =("delta_all",   "std"),
                   delta_early_mean=("delta_early", "mean"),
                   delta_early_std =("delta_early", "std"),
                   delta_late_mean =("delta_late",  "mean"),
                   delta_late_std  =("delta_late",  "std"),
                   rmse_ord_all   =("rmse_ordered",  "mean"),
                   rmse_shuf_all  =("rmse_shuffled", "mean"))
              .reset_index())

print("=== Ordered vs Shuffled: ΔRMSE = RMSE_ordered − RMSE_shuffled ===")
print("   (positive => shuffled is better;  negative => order helps)\n")
print(summary.to_string(index=False))

print("\n=== Reference points (RMSE) ===")
print(ref.groupby(["dataset", "L"])[["rmse_ordered", "rmse_ordered_early",
                                     "rmse_ordered_late"]]
        .agg(["mean", "std"]).to_string())